# Modèle ZÉRO : français + kirundi (Google Colab)

Le modèle part de **rien** : poids au hasard, nouveau tokenizer. Tout vient du dossier **`dataset`**
(textes, dialogues, tests et code), le même pour Colab, Kaggle et la machine.

| Étape | Ce qui se passe | Durée (GPU T4 gratuit) |
|---|---|---|
| 0 | Préparer : GPU, dossiers | 2 min |
| 1 | Télécharger les données publiques (Wikipédia FR, OpenAssistant, PIAF, Aya) | 30 à 60 min |
| 2 | Créer le tokenizer (32 000 morceaux, moitié kirundi, moitié français) | 15 min |
| 3 | **Lecture** : le modèle lit tous les textes | 12 à 15 h, en 3 ou 4 sessions (Colab coupe après quelques heures) |
| 4 | **Discussion** : il apprend à répondre, traduire, lire un document | 2 à 3 h |
| 5 | Tests sur des exemples jamais vus | 15 min |
| 6 | Essayer le modèle et récupérer le `.zip` pour le backend | |

Ces durées sont des **estimations**. La vraie vitesse s'affiche pendant l'entraînement (`s/pas` dans le journal).

**Avant de commencer** : glisse le dossier `datasets/dataset` dans **Mon Drive** (à la racine).
Menu **Exécution > Modifier le type d'exécution > GPU T4**.

**Si Colab coupe** : relance l'étape 0, puis la cellule de l'étape en cours. Tout reprend au dernier point sauvegardé dans Drive.

## Étape 0 : préparer Colab

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')

import os, shutil
DATASET = "/content/drive/MyDrive/dataset"        # lecture
TRAVAIL = "/content/drive/MyDrive/modele_zero"    # écriture (survit aux coupures)
# Le code est copié hors de Drive (plus rapide, et Drive reste propre)
CODE = "/content/code"
shutil.copytree(f"{DATASET}/code", CODE, dirs_exist_ok=True)
!pip install -q -r {CODE}/requirements.txt

In [ ]:
# 0.b  Commun à toutes les plateformes : variables lues par train.py ({DATASET} et {TRAVAIL} dans les configs)
os.environ["DATASET"], os.environ["TRAVAIL"] = DATASET, TRAVAIL
os.makedirs(f"{TRAVAIL}/journaux", exist_ok=True)
os.chdir(CODE)
J = f"{TRAVAIL}/journaux"

def fait(chemin):
    return os.path.exists(chemin)

def verifier(chemin, etape):
    if os.path.exists(chemin):
        print(f"OK : {etape} terminée -> {chemin}")
        return
    raise SystemExit(f"ÉCHEC : {etape} n'a pas produit {chemin}. Regarde {J}/{etape}.txt. "
                     "Ne passe PAS à l'étape suivante.")

def meilleur_modele():
    for ph in ("chat", "lecture"):
        for sous in ("finale", "meilleure", "derniere"):
            if os.path.exists(f"{TRAVAIL}/{ph}/{sous}/config.json"):
                return f"{TRAVAIL}/{ph}/{sous}"
    raise SystemExit("Aucun modèle trouvé : lance d'abord l'étape 3.")

for d in ("lecture_rn", "lecture_fr", "lecture_bilingue", "grammaire", "dialogues"):
    print(("OK      " if fait(f"{DATASET}/{d}") else "MANQUANT"), f"{DATASET}/{d}")
print()
for nom, chemin in [("1. Données publiques", f"{TRAVAIL}/publics/lecture_fr/FINI"),
                    ("2. Tokenizer", f"{TRAVAIL}/tokenizer/tokenizer.json"),
                    ("3. Lecture", f"{TRAVAIL}/lecture/finale"), ("4. Discussion", f"{TRAVAIL}/chat/finale")]:
    reprise = fait(chemin.replace("finale", "derniere"))
    print(f"{nom:22s}: " + ("FINI" if fait(chemin) else "en cours (reprise possible)" if reprise else "pas commencé"))

## Étape 1 : données publiques
Une source qui ne se télécharge pas est sautée, les autres continuent.

In [ ]:
os.chdir(CODE)
if not fait(f"{TRAVAIL}/publics/lecture_fr/FINI"):
    !python -u collecte/telecharger_publics.py --sortie {TRAVAIL}/publics --mots 150000000 2>&1 | grep -v "examples/s" | tee {J}/publics.txt
!ls {TRAVAIL}/publics/lecture_fr | head -3; wc -l {TRAVAIL}/publics/dialogues/*.jsonl

## Étape 2 : le tokenizer
Le kirundi est lu 10 fois et un seul fichier Wikipédia une fois : les deux langues ont à peu près **la même place**.
Regarde la vérification à la fin : les phrases kirundi doivent être coupées en morceaux qui ont du sens.

In [ ]:
os.chdir(CODE)
if not fait(f"{TRAVAIL}/tokenizer/tokenizer.json"):
    !python train_tokenizer.py --taille_vocab 32000 \
        --source "{DATASET}/lecture_rn/*.txt=10" --source "{DATASET}/lecture_bilingue/*.txt=3" \
        --source "{TRAVAIL}/publics/lecture_fr/wikipedia_fr_001.txt=1" --source "{DATASET}/lecture_fr/*.txt=1" \
        --sortie {TRAVAIL}/tokenizer 2>&1 | tee {J}/tokenizer.txt
verifier(f"{TRAVAIL}/tokenizer/tokenizer.json", "tokenizer")

## Étape 3 : lecture (la plus longue)
**Si ça coupe** : relance l'étape 0, puis cette cellule. Ça reprend tout seul.
Si *CUDA out of memory* : dans `code/configs/zero_pretrain.yaml`, mets `taille_lot: 4` et `accumulation: 32`.

In [ ]:
os.chdir(CODE)
if not fait(f"{TRAVAIL}/lecture/finale"):
    !python -u train.py --config configs/zero_pretrain.yaml 2>&1 | grep --line-buffered -vE 'it/s\]|examples/s' | tee -a {J}/lecture.txt
verifier(f"{TRAVAIL}/lecture/finale", "lecture")

## Étape 4 : discussion (répondre, traduire, lire un document)

In [ ]:
os.chdir(CODE)
if not fait(f"{TRAVAIL}/chat/finale"):
    !python -u train.py --config configs/zero_chat.yaml 2>&1 | grep --line-buffered -vE 'it/s\]|examples/s' | tee -a {J}/chat.txt
verifier(f"{TRAVAIL}/chat/finale", "chat")

## Étape 5 : tests
**exact** = réponse parfaite, **chrF** = ressemblance (0 à 100, la mesure habituelle des traductions).
Seules les catégories **« NON (vraie preuve) »** montrent si le modèle a compris : il ne les a jamais vues.
La plus importante : **traductions_bible** (versets jamais vus, dans les deux sens).

In [ ]:
os.chdir(CODE)
!python -u evaluer.py --modele {meilleur_modele()} --tests tests/dialogues_test.jsonl --max 100 --montrer 3 \
    --sortie {J}/resultats_tests.jsonl 2>&1 | tee {J}/notes_tests.txt

## Étape 6 : essayer le modèle

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
appareil = "cuda" if torch.cuda.is_available() else "cpu"
dossier = meilleur_modele()
tok = AutoTokenizer.from_pretrained(dossier)
model = AutoModelForCausalLM.from_pretrained(dossier).to(appareil).eval()
historique = ""

def demander(question, garder=True):
    """Pose une question. garder=True : le modèle se souvient des tours d'avant."""
    global historique
    prompt = f"{historique}<|user|> {question} <|assistant|>"
    ids = tok(prompt, return_tensors="pt", return_token_type_ids=False).to(appareil)
    out = model.generate(**ids, max_new_tokens=150, do_sample=True, temperature=0.3, top_p=0.9,
                         eos_token_id=tok.eos_token_id, pad_token_id=tok.pad_token_id)
    reponse = tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    if garder:
        historique = f"{prompt} {reponse}<|endoftext|>"
    return reponse

print(dossier)
for q in ["Bonjour", "Bwakeye.", "N'amaki?", "Comment dit-on merci en kirundi ?", "Qui es-tu ?",
          "Traduis en français : Umwana n'umukobwa."]:
    print(q, "->", demander(q, garder=False))

In [ ]:
# Une vraie conversation (historique = "" pour recommencer)
historique = ""
print(demander("Bonjour"))
print(demander("Comment dit-on au revoir en kirundi ?"))
print(demander("Merci"))

## Étape 7 : le `.zip` pour le backend
À mettre dans `litgpt/models/local/`, puis `python scripts/convert_local_model.py <dossier>`.

In [ ]:
dossier_modele = meilleur_modele()
!cd "{dossier_modele}" && zip -qr /content/modele_zero_fr_rn.zip . -x "etat_entrainement.pt" && ls -lh /content/modele_zero_fr_rn.zip
!cp /content/modele_zero_fr_rn.zip {TRAVAIL}/
from google.colab import files
files.download("/content/modele_zero_fr_rn.zip")